In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, DateType, TimestampType, LongType
)
from datetime import datetime
from zoneinfo import ZoneInfo
import uuid

# Keep log timestamps consistent with the project timezone.
spark.conf.set("spark.sql.session.timeZone", "Asia/Kolkata")

In [0]:
CATALOG = "Stream_Bank_Processing"
SCHEMA = "bronze"
BASE_PATH = "/Volumes/private_volume/volume/raw_files"

CUSTOMER_INITIAL_PATH = f"{BASE_PATH}/customers/customers.csv"
ACCOUNT_INITIAL_PATH = f"{BASE_PATH}/accounts/accounts.csv"
TRANSACTION_INITIAL_PATH = f"{BASE_PATH}/transactions/transactions.csv"

CUSTOMER_INCOMING_PATH = f"{BASE_PATH}/incoming/customers"
ACCOUNT_INCOMING_PATH = f"{BASE_PATH}/incoming/accounts"
TRANSACTION_INCOMING_PATH = f"{BASE_PATH}/incoming/transactions"

# Use the existing actual folder layout under raw_files.
CUSTOMER_SCHEMA_PATH = f"{BASE_PATH}/schemas/bronze/customers"
ACCOUNT_SCHEMA_PATH = f"{BASE_PATH}/schemas/bronze/accounts"
TRANSACTION_SCHEMA_PATH = f"{BASE_PATH}/schemas/bronze/transactions"

CUSTOMER_CHECKPOINT = f"{BASE_PATH}/checkpoints/bronze/customers"
ACCOUNT_CHECKPOINT = f"{BASE_PATH}/checkpoints/bronze/accounts"
TRANSACTION_CHECKPOINT = f"{BASE_PATH}/checkpoints/bronze/transactions"

CUSTOMER_TABLE = f"{CATALOG}.{SCHEMA}.customers"
ACCOUNT_TABLE = f"{CATALOG}.{SCHEMA}.accounts"
TRANSACTION_TABLE = f"{CATALOG}.{SCHEMA}.transactions"
LOG_TABLE = f"{CATALOG}.{SCHEMA}.ingestion_log"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")



In [0]:
CUSTOMER_SCHEMA = StructType([
    StructField("customer_id", StringType(), False),
    StructField("name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("dob", DateType(), True),
    StructField("city", StringType(), True),
    StructField("segment", StringType(), True),
    StructField("kyc_status", StringType(), True),
    StructField("credit_score", IntegerType(), True),
    StructField("annual_income", DoubleType(), True),
    StructField("join_date", DateType(), True),
    StructField("operation", StringType(), True),
    StructField("event_timestamp", TimestampType(), True)
])

ACCOUNT_SCHEMA = StructType([
    StructField("account_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("account_type", StringType(), True),
    StructField("branch", StringType(), True),
    StructField("open_date", DateType(), True),
    StructField("status", StringType(), True),
    StructField("balance", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("operation", StringType(), True),
    StructField("event_timestamp", TimestampType(), True)
])

TRANSACTION_SCHEMA = StructType([
    StructField("txn_id", StringType(), False),
    StructField("account_id", StringType(), False),
    StructField("txn_type", StringType(), True),
    StructField("txn_timestamp", TimestampType(), True),
    StructField("channel", StringType(), True),
    StructField("merchant_category", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("fraud_flag", StringType(), True),
    StructField("operation", StringType(), True),
    StructField("event_timestamp", TimestampType(), True)
])
CUSTOMER_INCOMING_SCHEMA = StructType([
    StructField("customer_id", StringType(), False),
    StructField("name", StringType(), True),
    StructField("gender", StringType(), True),
])

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
    batch_id STRING,
    table_name STRING,
    load_type STRING,
    source_path STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    records_processed BIGINT,
    status STRING,
    error_message STRING
)
USING DELTA
""")

In [0]:
def now_ist():
    """Return current India time as a naive datetime for Spark TimestampType."""
    return datetime.now(ZoneInfo("Asia/Kolkata")).replace(tzinfo=None)


def write_ingestion_log(
    batch_id,
    table_name,
    load_type,
    source_path,
    start_time,
    end_time,
    records_processed,
    status,
    error_message=None
):
    """Append one ingestion outcome to the Bronze ingestion log."""
    log_schema = StructType([
        StructField("batch_id", StringType(), False),
        StructField("table_name", StringType(), False),
        StructField("load_type", StringType(), False),
        StructField("source_path", StringType(), True),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("records_processed", LongType(), True),
        StructField("status", StringType(), False),
        StructField("error_message", StringType(), True)
    ])

    row = [(
        batch_id,
        table_name,
        load_type,
        source_path,
        start_time,
        end_time,
        int(records_processed) if records_processed is not None else 0,
        status,
        str(error_message)[:4000] if error_message else None
    )]

    log_df = spark.createDataFrame(row, schema=log_schema)
    log_df.write.format("delta").mode("append").saveAsTable(LOG_TABLE)


def add_bronze_metadata(df, load_type, batch_id):
    """Add lineage and ingestion metadata without changing source attributes."""
    return (
        df.withColumn("source_file", F.input_file_name())
          .withColumn("ingestion_timestamp", F.current_timestamp())
          .withColumn("load_type", F.lit(load_type))
          .withColumn("batch_id", F.lit(batch_id))
    )

In [0]:

def initial_load_csv(source_path, source_schema, table_name):
    batch_id = f"initial_{table_name.split('.')[-1]}_{uuid.uuid4().hex[:12]}"
    start_time = now_ist()
    try:
        if spark.catalog.tableExists(table_name):
            print(f"SKIPPED initial load: {table_name} already exists")
            return

        source_df = (
            spark.read.format("csv")
            .option("header", True)
            .option("mode", "FAILFAST")
            .schema(source_schema)
            .load(source_path)
        )

        # Initial CSVs commonly do not have CDC fields; add them as NULL.
        for col_name, col_type in [("operation", "string"), ("event_timestamp", "timestamp")]:
            if col_name not in source_df.columns:
                source_df = source_df.withColumn(col_name, F.lit(None).cast(col_type))

        enriched = (
            source_df
            .withColumn("source_file", F.lit(source_path))
            .withColumn("ingestion_timestamp", F.current_timestamp())
            .withColumn("load_type", F.lit("INITIAL"))
            .withColumn("batch_id", F.lit(batch_id))
        )

        count = enriched.count()
        enriched.write.format("delta").mode("append").saveAsTable(table_name)

        write_ingestion_log(
            batch_id, table_name, "INITIAL", source_path,
            start_time, now_ist(), count, "SUCCESS"
        )
        print(f"INITIAL SUCCESS: {table_name}, rows={count}")

    except Exception as exc:
        try:
            write_ingestion_log(
                batch_id, table_name, "INITIAL", source_path,
                start_time, now_ist(), 0, "FAILED", str(exc)
            )
        except Exception as log_exc:
            print(f"Could not write failure log: {log_exc}")
        raise

# Initial load executes before incremental streams.
initial_load_csv(CUSTOMER_INITIAL_PATH, CUSTOMER_SCHEMA, CUSTOMER_TABLE)
initial_load_csv(ACCOUNT_INITIAL_PATH, ACCOUNT_SCHEMA, ACCOUNT_TABLE)
initial_load_csv(TRANSACTION_INITIAL_PATH, TRANSACTION_SCHEMA, TRANSACTION_TABLE)


In [0]:
def read_autoloader_stream(incoming_path, schema, schema_path):
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.schemaEvolutionMode", "failOnNewColumns")
        .option("header", True)
        .option("mode", "FAILFAST")
        .schema(schema)
        .load(incoming_path)
    )


In [0]:

def read_autoloader_stream(
    source_path,
    schema,
    schema_location
):
    # Convert StructType fields into Auto Loader schema hints
    schema_hints = ", ".join(
        f"{field.name} {field.dataType.simpleString()}"
        for field in schema.fields
    )

    stream_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_location)
        .option(
            "cloudFiles.schemaEvolutionMode",
            "addNewColumns"
        )
        .option(
            "cloudFiles.schemaHints",
            schema_hints
        )
        .option("header", "true")
        .option("mode", "FAILFAST")
        .load(source_path)
    )

    return stream_df
    

def make_batch_processor(table_name, source_path, label):
    def process_batch(batch_df, stream_batch_id):
        batch_id = f"{label}_stream_{stream_batch_id}"
        start_time = now_ist()
        try:
            # Capture file lineage from Auto Loader metadata (UC-supported).
            batch_df = batch_df.withColumn(
                "source_file", F.col("_metadata.file_path")
            )

            enriched = (
                batch_df
                .withColumn("ingestion_timestamp", F.current_timestamp())
                .withColumn("load_type", F.lit("INCREMENTAL"))
                .withColumn("batch_id", F.lit(batch_id))
            )

            record_count = enriched.count()
            if record_count > 0:
                enriched.write.format("delta").mode("append").saveAsTable(table_name)

            write_ingestion_log(
                batch_id, table_name, "INCREMENTAL", source_path,
                start_time, now_ist(), record_count, "SUCCESS"
            )
            print(f"INCREMENTAL SUCCESS: {table_name}, batch={stream_batch_id}, rows={record_count}")

        except Exception as exc:
            try:
                write_ingestion_log(
                    batch_id, table_name, "INCREMENTAL", source_path,
                    start_time, now_ist(), 0, "FAILED", str(exc)
                )
            except Exception as log_exc:
                print(f"Failure logging failed for {table_name}: {log_exc}")
            raise
    return process_batch

def start_autoloader(incoming_path, schema, schema_path, checkpoint_path,
                     table_name, label):
    stream_df = read_autoloader_stream(incoming_path, schema, schema_path)
    processor = make_batch_processor(table_name, incoming_path, label)

    return (
        stream_df.writeStream
        .foreachBatch(processor)
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .start()
    )


In [0]:
customer_query = start_autoloader(
    CUSTOMER_INCOMING_PATH, CUSTOMER_SCHEMA,
    CUSTOMER_SCHEMA_PATH, CUSTOMER_CHECKPOINT,
    CUSTOMER_TABLE, "customer"
)

account_query = start_autoloader(
    ACCOUNT_INCOMING_PATH, ACCOUNT_SCHEMA,
    ACCOUNT_SCHEMA_PATH, ACCOUNT_CHECKPOINT,
    ACCOUNT_TABLE, "account"
)

transaction_query = start_autoloader(
    TRANSACTION_INCOMING_PATH, TRANSACTION_SCHEMA,
    TRANSACTION_SCHEMA_PATH, TRANSACTION_CHECKPOINT,
    TRANSACTION_TABLE, "transaction"
)

customer_query.awaitTermination()
account_query.awaitTermination()
transaction_query.awaitTermination()

In [0]:
print("Customers:", spark.table(CUSTOMER_TABLE).count())
print("Accounts:", spark.table(ACCOUNT_TABLE).count())
print("Transactions:", spark.table(TRANSACTION_TABLE).count())

print("Recent ingestion log:")
display(
    spark.table(LOG_TABLE)
    .orderBy(F.col("end_time").desc())
    .limit(20)
)

print("Latest customer rows:")
display(
    spark.table(CUSTOMER_TABLE)
    .orderBy(F.col("ingestion_timestamp").desc())
    .limit(10)
)

In [0]:
%sql
select count(*) from stream_bank_processing.bronze.transactions

In [0]:
%sql
select * from stream_bank_processing.bronze.accounts

In [0]:
%sql
SELECT 'customers' AS entity, COUNT(*) AS total
FROM Stream_Bank_Processing.bronze.customers

UNION ALL

SELECT 'accounts', COUNT(*)
FROM Stream_Bank_Processing.bronze.accounts

UNION ALL

SELECT 'transactions', COUNT(*)
FROM Stream_Bank_Processing.bronze.transactions;

In [0]:
%sql
SELECT
    _metadata.file_path AS source_file,
    COUNT(*) AS records
FROM Stream_Bank_Processing.bronze.customers
GROUP BY _metadata.file_path;

In [0]:
%sql
SELECT
    _metadata.file_path AS source_file,
    COUNT(*) AS records
FROM Stream_Bank_Processing.bronze.accounts
GROUP BY _metadata.file_path;

In [0]:
%sql
DESCRIBE HISTORY Stream_Bank_Processing.bronze.accounts;

In [0]:
%sql
SELECT *
FROM Stream_Bank_Processing.bronze.accounts
-- order by end_time desc;

In [0]:
%sql
select * from stream_bank_processing.bronze.ingestion_log
order by end_time desc